<a href="https://colab.research.google.com/github/ragiokay/ML-2026-HW7/blob/main/ML2026_Spring_HW7_Model_Merging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ML2026 Spring HW7 — Model Merging**

在這份作業中，將實作 **Model Merging** 。

我們會使用兩個 7B 模型：
- `augmxnt/shisa-gamma-7b-v1`（擅長日文）
- `WizardLMTeam/WizardMath-7B-V1.1`（擅長數學推理）

**❗重要規則：**

- **請勿使用上述兩個模型以外的任何其他模型。**
- 你可以自由使用任何套件（不限於 mergekit）或自己撰寫的演算法來進行 merging。
- **Merging 的嚴格定義：合併後的模型總參數量必須與單一 base model 相同。** 換言之，MoE（Mixture of Experts）、Ensemble、Stacking 等會使 inference 時可使用的參數量增加的做法，皆視為違規。

你的目標是：
1. 先觀察兩個 base model 各自的能力
2. 嘗試各種 merge 方法，把兩個模型合併
3. 在日文數學題上測試合併後的模型表現，並將結果上傳到 judgeboi

# **Section 0: Preparing**

## Mount your notebook to your Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Activate GPU
Model Merging 與 Inference 都需要 GPU，請務必啟用。

### **MUST READ**:

Colab does **NOT** guarantee the GPU access for free user ([ref](https://research.google.com/colaboratory/faq.html#idle-timeouts)). It is possible you get a message saying "Cannot connect to GPU backend" which means there are no enough GPU resources for you now. When this happens, you may need to **wait for one (or more) day or login different Google account to do the homework**.

### Enable GPU

1. Click on "Runtime" (or "執行階段") in the header.
2. Click on "Change runtime type" (or "變更執行階段類型") in the drop menu.
3. Select "T4 GPU" and save. (You can select "A100 GPU" or "V100 GPU" if you have Colab Pro)

## Check the status of your GPU

In [ ]:
!nvidia-smi

## Install Packages & Download Data

安裝 `mergekit`（模型合併工具）及下載日文數學題目檔案。

In [ ]:
# Install mergekit
!pip install -q mergekit

# Install / upgrade gdown for Google Drive download
!pip install -U -q gdown

# Download the Japanese math question file
!gdown 1_9OAxDwQgz1jUnlhHlkk3lLCTpY1Ib8Q -O /content/jp_math_question.json
# Verify download
import json
with open("/content/jp_math_question.json", "r") as f:
    questions = json.load(f)
print(f"成功載入 {len(questions)} 題日文數學題目！")
print(f"範例題目: {questions[0]['question'][:80]}...")
print(f"範例答案: {questions[0]['answer_number']}")

# **Section 1: Explore Base Models — 觀察兩個 Base Model 的能力**
## 若時間不足可以先跳到Section 2

在合併之前，我們先分別載入兩個 base model，觀察它們各自的強項與弱項。

- **shisa-gamma-7b-v1**：以日文能力見長的模型
- **WizardMath-7B-V1.1**：以數學推理見長的模型

我們會各丟一個 **日文問題** 和一個 **數學問題** 給兩個模型，直覺感受它們的差異。

## 1.1 定義推論用的 helper function

建立通用的推論與評估函數，後續兩個模型都會用到。

In [ ]:
def generate_response(model, tokenizer, prompt, max_new_tokens=256):
    """通用推論函數：給定模型與 prompt，回傳生成結果"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1
        )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_output[len(prompt):].strip()

In [ ]:
import re

############################################################
# Prompt 模板
USER_PROMPT_TEMPLATE = (
    "次の数学の問題を解いて、最終的な数値の答えを提示してください。\n\n"
    "問題: {question}\n\n"
    "ステップバイステップで考えて、最後に『答え: [数値]』の形式で答えを提示してください。"
)

# ⛔ 嚴禁修改 USER_PROMPT_TEMPLATE！
#
# 本作業聚焦於 Model Merging 技術本身，
# 而非 Prompt Engineering，因此為確保所有同學的
# 評測條件一致，無論任何情況皆請勿修改以下 prompt。
# 違反此規則將視為違規。
############################################################


def extract_number(text):
    """從模型輸出中提取數字答案"""
    match = re.search(r'答え\s*[:：]\s*(\d+)', text)
    if match:
        return match.group(1)
    numbers = re.findall(r'\d+', text)
    return numbers[-1] if numbers else None

def evaluate_model(model, tokenizer, questions, model_name):
    """在日文數學題集上評估模型"""
    correct = 0
    results = []
    print(f"\n{'='*60}")
    print(f"📊 評估模型: {model_name}")
    print(f"{'='*60}\n")

    for i, item in enumerate(questions):
        prompt = USER_PROMPT_TEMPLATE.format(question=item['question'])
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                repetition_penalty=1.1
            )
        full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
        answer_part = full_output[len(prompt):].strip()
        predicted = extract_number(answer_part)
        expected = item["answer_number"]
        is_correct = predicted == expected
        if is_correct:
            correct += 1
        results.append({
            "id": i + 1, "correct": is_correct,
            "predicted": predicted, "expected": expected,
            "raw": answer_part,
        })
        mark = "✅" if is_correct else "❌"
        print(f"{mark} Q{i+1:02d} | 預測: {str(predicted):>6s} | 正解: {expected:>6s}")
        print(f"     | 題目: {item['question'][:60]}")
        print(f"     | raw: {answer_part[:120]}")
        print()

    print(f"{'='*50}")
    print(f"🎯 {model_name} 正確率: {correct}/{len(questions)} ({correct/len(questions)*100:.1f}%)")
    print(f"{'='*50}")
    return correct, results

## 1.2 測試 Base Model ①：shisa-gamma-7b-v1（日文模型）

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("載入 shisa-gamma-7b-v1 ...")
shisa_name = "augmxnt/shisa-gamma-7b-v1"
shisa_tokenizer = AutoTokenizer.from_pretrained(shisa_name)
shisa_model = AutoModelForCausalLM.from_pretrained(
    shisa_name, torch_dtype=torch.float16, device_map="auto"
)
print("shisa-gamma-7b-v1 載入完成！")

### 日文能力測試（shisa）

In [ ]:
jp_prompt = "日本の四季の特徴について、それぞれ簡潔に説明してください。"

print("=" * 60)
print("📝 日文 Prompt:", jp_prompt)
print("=" * 60)
print("\n🔵 【shisa-gamma-7b（日文模型）】的回答：")
print("-" * 40)
shisa_jp_ans = generate_response(shisa_model, shisa_tokenizer, jp_prompt)
print(shisa_jp_ans[:500])

### 數學能力測試（shisa）

In [ ]:
math_prompt = (
    "Solve the following math problem step by step.\n\n"
    "Problem: A store sells apples for $2 each and oranges for $3 each. "
    "If a customer buys 5 apples and 4 oranges, and pays with a $50 bill, "
    "how much change should the customer receive?\n\n"
    "Solution:"
)

print("=" * 60)
print("📝 Math Prompt:")
print(math_prompt)
print("=" * 60)
print("\n🔵 【shisa-gamma-7b（日文模型）】的回答：")
print("-" * 40)
shisa_math_ans = generate_response(shisa_model, shisa_tokenizer, math_prompt)
print(shisa_math_ans[:500])

print("\n" + "=" * 60)
print("💡 正確答案：$50 - (5×$2 + 4×$3) = $50 - $22 = $28")
print("=" * 60)

### 在日文數學題集上評估（shisa）

In [ ]:
with open("/content/jp_math_question.json", "r") as f:
    questions = json.load(f)

shisa_correct, shisa_results = evaluate_model(
    shisa_model, shisa_tokenizer, questions, "shisa-gamma-7b（日文模型）"
)

### 釋放 shisa 模型記憶體

In [ ]:
import gc

del shisa_model, shisa_tokenizer
gc.collect()
torch.cuda.empty_cache()
print("已釋放 shisa-gamma-7b 記憶體！")
!nvidia-smi --query-gpu=memory.used,memory.free --format=csv

## 1.3 測試 Base Model ②：WizardMath-7B-V1.1（數學模型）

In [ ]:
print("載入 WizardMath-7B-V1.1 ...")
wizard_name = "WizardLMTeam/WizardMath-7B-V1.1"
wizard_tokenizer = AutoTokenizer.from_pretrained(wizard_name)
wizard_model = AutoModelForCausalLM.from_pretrained(
    wizard_name, torch_dtype=torch.float16, device_map="auto"
)
print("WizardMath-7B-V1.1 載入完成！")

### 日文能力測試（WizardMath）

In [ ]:
print("=" * 60)
print("📝 日文 Prompt:", jp_prompt)
print("=" * 60)
print("\n🟠 【WizardMath-7B（數學模型）】的回答：")
print("-" * 40)
wizard_jp_ans = generate_response(wizard_model, wizard_tokenizer, jp_prompt)
print(wizard_jp_ans[:500])

### 數學能力測試（WizardMath）

In [ ]:
print("=" * 60)
print("📝 Math Prompt:")
print(math_prompt)
print("=" * 60)
print("\n🟠 【WizardMath-7B（數學模型）】的回答：")
print("-" * 40)
wizard_math_ans = generate_response(wizard_model, wizard_tokenizer, math_prompt)
print(wizard_math_ans[:500])

print("\n" + "=" * 60)
print("💡 正確答案：$50 - (5×$2 + 4×$3) = $50 - $22 = $28")
print("=" * 60)

### 在日文數學題集上評估（WizardMath）

In [ ]:
wizard_correct, wizard_results = evaluate_model(
    wizard_model, wizard_tokenizer, questions, "WizardMath-7B（數學模型）"
)

In [ ]:
del wizard_model, wizard_tokenizer
gc.collect()
torch.cuda.empty_cache()
print("已釋放 WizardMath-7B 記憶體！")
!nvidia-smi --query-gpu=memory.used,memory.free --format=csv

### 🔍 Base Model Baseline

記下兩個 base model 的正確率，之後合併模型時可以做比較：
- shisa-gamma-7b：擅長日文，但數學推理能力可能不足
- WizardMath-7B：擅長數學推理，但可能無法理解日文題意

理想的 merge 應該要能結合兩者的優勢！

# **Section 2: Model Merging — 合併模型**

## 2.1 撰寫 Merge 設定檔

以下是一個最簡單的 **50/50 Linear Merge**（線性插值合併）設定。

### ⚠️ 這是你的主要修改區域！

你可以嘗試不同的合併策略，例如：
- **linear**：最直覺的加權平均（調整 `weight` 比例）
- **slerp**：球面線性插值（調整 `t` 參數）
- **ties**：TIES-Merging（需設定 `density` 與 `weight`）
- **dare_ties**：DARE + TIES（需設定 `density`、`weight`）

請參考 [mergekit 文件](https://github.com/arcee-ai/mergekit?tab=readme-ov-file#merge-methods) 了解更多合併方法與參數。

In [ ]:
############################################################
# ⚠️ 【可修改區域 — TODO】
#
# 預設：Slerp Merge
# 修改底下 config 參數內容，可以更改之部分與詳情請見：
#
# 可嘗試的方向：
#   1. 調整 weight 比例（例如 0.7:0.3）
#   2. 改用 linear 方法
#   3. 改用 ties 方法
#   4. 改用 dare_ties 方法
#   5. 針對不同 layer 使用不同參數
#
# 提示：每次修改後需重新執行 Section 2 & 3

config = """
slices:
  - sources:
      - model: augmxnt/shisa-gamma-7b-v1
        layer_range: [0, 32]
      - model: WizardLMTeam/WizardMath-7B-V1.1
        layer_range: [0, 32]
merge_method: slerp
base_model: augmxnt/shisa-gamma-7b-v1
parameters:
  t:
    - filter: self_attn
      value: [0, 0.5, 0.3, 0.7, 1]
    - filter: mlp
      value: [1, 0.5, 0.7, 0.3, 0]
    - value: 0.5
dtype: float16
""".strip()


############################################################

with open("merge_config.yml", "w") as f:
    f.write(config)
print("設定檔寫入完成！內容如下：")
print("=" * 40)
print(config)
print("=" * 40)

In [ ]:
# 備存 merge_config.yml 到 Google Drive
import shutil, os
drive_backup_dir = "/content/drive/MyDrive/ML2026/HW7"
os.makedirs(drive_backup_dir, exist_ok=True)
shutil.copy("merge_config.yml", os.path.join(drive_backup_dir, "merge_config.yml"))
print(f"✅ merge_config.yml 已備存至 {drive_backup_dir}/merge_config.yml")

## 2.2 執行合併

使用 mergekit 進行模型合併（GPU 模式 + lazy unpickle 逐層載入以節省記憶體）。

由於模型大小為 7B 偏大，這個步驟根據不同方法可能需要花上30分鐘至數小時，請耐心等候。

In [ ]:
# 若先前有合併過的模型，先清除
!rm -rf ./merged_model

# 執行合併（GPU 模式 + lazy unpickle 逐層載入）
!mergekit-yaml merge_config.yml ./merged_model \
    --lazy-unpickle \
    --allow-crimes \
    --cuda

## 2.3 確認合併輸出

In [ ]:
import os

output_dir = "./merged_model"
if os.path.exists(output_dir):
    files = os.listdir(output_dir)
    total_size = sum(os.path.getsize(os.path.join(output_dir, f)) for f in files)
    print(f"\n✅ 合併完成！檔案數: {len(files)}, 總大小: {total_size / 1e9:.2f} GB")
    for f in sorted(files):
        size = os.path.getsize(os.path.join(output_dir, f))
        print(f"  {f}: {size / 1e6:.1f} MB")
else:
    print("❌ 合併失敗，請檢查上方錯誤訊息")

# **Section 3: Inference — 測試合併後的模型**

## 3.1 載入合併後的模型

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import re

# Install sentencepiece if not already installed, as it's often needed for tokenizers.
!pip install -U sentencepiece tiktoken
output_dir = "./merged_model"

print("載入合併後的模型中...")
tokenizer = AutoTokenizer.from_pretrained(output_dir)
model = AutoModelForCausalLM.from_pretrained(
    output_dir, torch_dtype=torch.float16, device_map="auto"
)
print("載入完成！")

prompt = "東京の人口は"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=50, do_sample=False)
print(f"測試輸出: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")

## 3.2 在日文數學題上測試合併模型

使用與 Section 1 相同的題目與評估方式。
Inference 時間可能會需要1-2小時，請耐心等待

In [ ]:
import json
import re

# 1. 載入題目
with open("/content/jp_math_question.json", "r", encoding="utf-8") as f:
    questions = json.load(f)

############################################################
# 2. Prompt 模板
USER_PROMPT_TEMPLATE = (
    "次の数学の問題を解いて、最終的な数値の答えを提示してください。\n\n"
    "問題: {question}\n\n"
    "ステップバイステップで考えて、最後に『答え: [数値]』の形式で答えを提示してください。"
)

# ⛔ 嚴禁修改 USER_PROMPT_TEMPLATE！
# 本作業聚焦於 Model Merging 技術本身，
# 而非 Prompt Engineering，因此為確保所有同學的
# 評測條件一致，無論任何情況皆請勿修改以下 prompt。
# 違反此規則將視為違規。
############################################################

# 3. 推論 + 評分
def extract_number(text):
    match = re.search(r'答え\s*[:：]\s*(\d+)', text)
    if match:
        return match.group(1)
    numbers = re.findall(r'\d+', text)
    return numbers[-1] if numbers else None

correct = 0
results = []

print(f"{'='*60}")
print("📊 評估合併模型 (Merged Model)")
print(f"{'='*60}\n")

for i, item in enumerate(questions):
    prompt = USER_PROMPT_TEMPLATE.format(question=item["question"])

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            repetition_penalty=1.1
        )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer_part = full_output[len(prompt):].strip()
    predicted = extract_number(answer_part)
    expected = item["answer_number"]
    is_correct = predicted == expected

    if is_correct:
        correct += 1

    results.append({
        "id": i + 1,
        "correct": is_correct,
        "predicted": predicted,
        "expected": expected,
        "raw": answer_part,
    })

    mark = "✅" if is_correct else "❌"
    print(f"{mark} Q{i+1:02d} | 預測: {str(predicted):>6s} | 正解: {expected:>6s}")
    print(f"     | 題目: {item['question'][:60]}")
    print(f"     | raw: {answer_part[:120]}")
    print()

# 4. 總結
print(f"{'='*50}")
print(f"🎯 Merged Model 正確率: {correct}/{len(questions)} ({correct/len(questions)*100:.1f}%)")
print(f"{'='*50}")

# ==========================================
# 5. 儲存結果為 JSON
# ==========================================
submission_path = "/content/submission.json"

# 若你只想保留 id 和 raw
submission_data = [
    {
        "id": r["id"],
        "raw": r["raw"]
    }
    for r in results
]

with open(submission_path, "w", encoding="utf-8") as f:
    json.dump(submission_data, f, ensure_ascii=False, indent=2)

print(f"\n📁 結果已儲存至 {submission_path}")

In [ ]:
# 備存 submission.json 到 Google Drive
import shutil, os
drive_backup_dir = "/content/drive/MyDrive/ML2026/HW7"
os.makedirs(drive_backup_dir, exist_ok=True)
shutil.copy(submission_path, os.path.join(drive_backup_dir, "submission.json"))
print(f"✅ submission.json 已備存至 {drive_backup_dir}/submission.json")